In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("agri_merged_raw.csv")
df = pd.read_csv(data_path)

### QUESTION 3

In [10]:

data_dictionary = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Meaning": [
        "Unique row identifier",
        "Date of observation",
        "Year of harvest",
        "Country or territory name",
        "3-letter ISO country code",
        "World sub-region",
        "Type of crop grown",
        "Land area harvested",
        "Total crop harvested",
        "Crop yield per hectare",
        "Crop yield in hectograms",
        "Average annual rainfall",
        "Total pesticides used",
        "Average annual temperature"
    ],
    "Units / Categories": [
        "Integers (1 to 25,432)",
        "YYYY-MM-DD (1990-01-01 to 2013-01-01)",
        "Year (1990 to 2013)",
        "204 countries",
        "201 codes (e.g., KEN, ZWE)",
        "26 regions (e.g., Eastern Africa)",
        "10 crops (e.g., Maize (corn), Wheat, Rice)",
        "Hectares (ha)",
        "Metric tonnes (t)",
        "Kilograms per hectare (kg/ha)",
        "Hectograms per hectare (hg/ha)",
        "Millimeters (mm)",
        "Metric tonnes (t)",
        "Degrees Celsius (°C)"
    ],
    "Expected Missing Values": [
        "None (0%)",
        "None (0%)",
        "None (0%)",
        "None (0%)",
        "None (0%)",
        "None (0%)",
        "None (0%)",
        "Very low (< 1%)",
        "None (0%)",
        "Very low (< 1%)",
        "Moderate (due to merged tables)",
        "Moderate (weather station coverage)",
        "Moderate (reporting coverage)",
        "Moderate (weather station coverage)"
    ]
})

# Export directly to data_dictionary.md 
headers = "| " + " | ".join(data_dictionary.columns) + " |"
separator = "| " + " | ".join(["---"] * len(data_dictionary.columns)) + " |"
rows = ["| " + " | ".join(str(val) for val in row) + " |" for row in data_dictionary.values]

markdown_text = "# Data Dictionary\n\n" + "\n".join([headers, separator] + rows)

Path("data_dictionary.md").write_text(markdown_text, encoding="utf-8")
print("Saved data_dictionary.md successfully!")

Saved data_dictionary.md successfully!


In [11]:
from IPython.display import Markdown, display

display(Markdown(Path("data_dictionary.md").read_text(encoding="utf-8")))

# Data Dictionary

| Column Name | Data Type | Meaning | Units / Categories | Expected Missing Values |
| --- | --- | --- | --- | --- |
| record_id | int64 | Unique row identifier | Integers (1 to 25,432) | None (0%) |
| observation_date | str | Date of observation | YYYY-MM-DD (1990-01-01 to 2013-01-01) | None (0%) |
| year | int64 | Year of harvest | Year (1990 to 2013) | None (0%) |
| country | str | Country or territory name | 204 countries | None (0%) |
| iso3 | str | 3-letter ISO country code | 201 codes (e.g., KEN, ZWE) | None (0%) |
| region | str | World sub-region | 26 regions (e.g., Eastern Africa) | None (0%) |
| crop | str | Type of crop grown | 10 crops (e.g., Maize (corn), Wheat, Rice) | None (0%) |
| plot_area_ha | float64 | Land area harvested | Hectares (ha) | Very low (< 1%) |
| production_tonnes | float64 | Total crop harvested | Metric tonnes (t) | None (0%) |
| yield_kg_ha | float64 | Crop yield per hectare | Kilograms per hectare (kg/ha) | Very low (< 1%) |
| yield_hg_ha | float64 | Crop yield in hectograms | Hectograms per hectare (hg/ha) | Moderate (due to merged tables) |
| rainfall_mm | float64 | Average annual rainfall | Millimeters (mm) | Moderate (weather station coverage) |
| pesticides_tonnes | float64 | Total pesticides used | Metric tonnes (t) | Moderate (reporting coverage) |
| temperature_c | float64 | Average annual temperature | Degrees Celsius (°C) | Moderate (weather station coverage) |

### QUESTION 4

In [2]:
# Checking null counts and percentages
null_summary = pd.DataFrame({
    "Null_Count": df.isnull().sum(),
    "Null_Pct": (df.isnull().sum() / len(df) * 100).round(2)
})
null_summary

,Null_Count,Null_Pct
record_id,0,0.00
observation_date,0,0.00
year,0,0.00
country,0,0.00
iso3,0,0.00
region,0,0.00
crop,0,0.00
plot_area_ha,2,0.01
production_tonnes,0,0.00
yield_kg_ha,213,0.84


In [3]:
# Checking for exact duplicate rows and duplicate IDs
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate record_id:", df["record_id"].duplicated().sum())

Duplicate rows: 0
Duplicate record_id: 0


In [4]:
# Checking numeric columns for negative values and zeros
num_cols = ["plot_area_ha", "production_tonnes", "yield_kg_ha", "yield_hg_ha", "rainfall_mm", "pesticides_tonnes", "temperature_c"]

for col in num_cols:
    neg = (df[col] < 0).sum()
    zero = (df[col] == 0).sum()
    print(f"{col}: negative={neg}, zero={zero}, min={df[col].min()}, max={df[col].max()}")

plot_area_ha: negative=0, zero=242, min=0.0, max=48178000.0
production_tonnes: negative=0, zero=213, min=0.0, max=351271870.0
yield_kg_ha: negative=0, zero=1, min=0.0, max=55485.5
yield_hg_ha: negative=0, zero=0, min=50.0, max=501412.0
rainfall_mm: negative=0, zero=0, min=51.0, max=3240.0
pesticides_tonnes: negative=0, zero=0, min=0.04, max=367778.0
temperature_c: negative=0, zero=0, min=1.3, max=30.42


In [5]:
# Checking text columns for extra spaces or casing differences
for col in ["country", "region", "crop"]:
    raw = df[col].nunique()
    cleaned = df[col].str.strip().str.lower().nunique()
    print(f"{col}: raw={raw}, stripped_lower={cleaned}")

df["crop"].value_counts()

country: raw=204, stripped_lower=204
region: raw=26, stripped_lower=26
crop: raw=10, stripped_lower=10


crop
Maize (corn)                     3838
Potatoes                         3646
Wheat                            2923
Rice                             2811
Sweet potatoes                   2664
Sorghum                          2537
Cassava, fresh                   2368
Soya beans                       2193
Yams                             1305
Plantains and cooking bananas    1147
Name: count, dtype: int64

In [6]:
# Checking date parsing without modifying the original column
parsed_dates = pd.to_datetime(df["observation_date"], errors="coerce")
print("Unparseable dates:", parsed_dates.isnull().sum())
print("Date range:", parsed_dates.min(), "to", parsed_dates.max())

Unparseable dates: 0
Date range: 1990-01-01 00:00:00 to 2013-01-01 00:00:00


### QUESTION 5

In [7]:
# Reusable function 1: Validate dataset shape and required columns
def validate_columns(dataframe, expected_cols):
    missing = [col for col in expected_cols if col not in dataframe.columns]
    print("Rows:", dataframe.shape[0], "| Columns:", dataframe.shape[1])
    print("Meets 15,000 rows & 10 cols:", dataframe.shape[0] >= 15000 and dataframe.shape[1] >= 10)
    print("Missing columns:", missing)

# Reusable function 2: Summarize missingness across all columns
def summarize_missingness(dataframe):
    return pd.DataFrame({
        "Dtype": dataframe.dtypes,
        "Null_Count": dataframe.isnull().sum(),
        "Null_Pct": (dataframe.isnull().sum() / len(dataframe) * 100).round(2),
        "Unique_Values": dataframe.nunique()
    })

required_cols = [
    "record_id", "observation_date", "year", "country", "iso3", "region",
    "crop", "plot_area_ha", "production_tonnes", "yield_kg_ha",
    "yield_hg_ha", "rainfall_mm", "pesticides_tonnes", "temperature_c"
]

validate_columns(df, required_cols)
summarize_missingness(df)

Rows: 25432 | Columns: 14
Meets 15,000 rows & 10 cols: True
Missing columns: []


,Dtype,Null_Count,Null_Pct,Unique_Values
record_id,int64,0,0.00,25432
observation_date,str,0,0.00,24
year,int64,0,0.00,24
country,str,0,0.00,204
iso3,str,0,0.00,201
region,str,0,0.00,26
crop,str,0,0.00,10
plot_area_ha,float64,2,0.01,16150
production_tonnes,float64,0,0.00,19771
yield_kg_ha,float64,213,0.84,20542


### QUESTION 6
### Data Quality Audit Table

| Check | Output Result | Status |
| :--- | :--- | :--- |
| **Dataset Size** | 25,432 rows × 14 columns | Pass ($\ge 15,000 \times 10$) |
| **Missing Values** | `plot_area_ha`: 2 (0.01%); `yield_kg_ha`: 213 (0.84%); `yield_hg_ha`, `rainfall_mm`, `pesticides_tonnes`, `temperature_c`: 12,328 (48.47%) each | Documented for Week 3 |
| **Duplicate Rows** | 0 exact duplicate rows; 0 duplicate `record_id` values | Clean |
| **Numeric Ranges** | 0 negative values; 242 rows have `plot_area_ha == 0`; 213 rows have `production_tonnes == 0` | Flag for Week 3 |
| **Categories** | 10 crops, 26 regions, 204 countries (no trailing spaces or casing mismatches) | Clean |
| **Date Parsing** | 0 unparseable dates; spans `1990-01-01` to `2013-01-01` | Clean (convert dtype in Week 3) |

---

### Five Initial Observations

1. **Dataset Size & Structure:** `df.shape` confirms **25,432 rows and 14 columns** with zero duplicate rows, meeting the $15,000 \times 10$ minimum requirement.
2. **Time Coverage:** `observation_date` and `year` span **24 years** from `1990-01-01` to `2013-01-01`.
3. **Geographic & Crop Spread:** The data covers **204 countries across 26 regions** and **10 crops**, led by `Maize (corn)` (3,838 rows) and `Potatoes` (3,646 rows), with `Eastern Africa` (2,839 rows) and `Western Africa` (2,594 rows) having the largest regional counts.
4. **Skewed Crop Yields:** In `df.describe()`, `yield_kg_ha` has a mean of **6,873.87 kg/ha** versus a median (`50%`) of **3,955.40 kg/ha** (max `55,485.50 kg/ha`), showing that high-yielding root and tuber crops pull the overall mean upward.
5. **Climate Measurement Ranges:** Across the 13,104 rows with weather records, `rainfall_mm` ranges from **51.0 mm to 3,240.0 mm** (mean `1,154.18 mm`) and `temperature_c` ranges from **1.30°C to 30.42°C** (mean `19.82°C`).

---

### Three Potential Data-Quality Issues

1. **48.47% Missing Climate & Pesticide Records:** Exactly **12,328 rows** are missing `rainfall_mm`, `temperature_c`, `pesticides_tonnes`, and `yield_hg_ha` simultaneously. Dropping all rows with nulls in Week 3 would leave only 13,101 rows (below the 15,000-row requirement), so we must retain all 25,432 rows for crop/region comparisons and filter only when analyzing climate variables.
2. **Zero Plot Areas & Missing Yields:** **242 rows** have `plot_area_ha == 0.0`, which causes **213 missing values** in `yield_kg_ha`. Additionally, `yield_kg_ha` does not equal `yield_hg_ha / 10` on 1,375 rows, so these rows need verification and flagging in Week 3.
3. **String Date Format & Historical Countries:** `observation_date` is currently stored as text (`str`) instead of `datetime64`, and 3 `iso3` codes (`CZE`, `ETH`, `SDN`) map to 6 country names due to historical splits (e.g., `Sudan` and `Sudan (former)`).